# run_pipeline_kaggle — ĐO trên dev, KHÔNG nộp bài

Notebook này chỉ để **đo `recall@5` nội bộ** trên `dev_150_locked.json` (150 câu cắt từ train).

> **Không build submission ở đây.** Bài nộp thật do `finalAnswer_run.ipynb` sinh, từ
> `Input/public-official.json`. Lần nộp 09/08 hỏng đúng vì notebook này từng build
> submission từ dev — phần đó đã được xoá hẳn thay vì vá.

D bàn giao candidate ĐÃ CHUNK (top-3 chunk/document, kèm `bm25_score`), nên không cần
load corpus. Bật **Internet: On** + **Accelerator: GPU T4 x2** trong Settings.


## Cài package + trỏ vào dataset


In [ ]:
!pip install -q sentence-transformers

import sys
INPUT_DIR = "/kaggle/input/project-ir"   # đổi đúng slug thật -- xem !ls /kaggle/input
sys.path.append(INPUT_DIR)
!ls /kaggle/input


In [ ]:
import json, os
from pathlib import Path

from metrics import evaluate, hard_cases
from rerank import load_reranker
from rerank_from_d import score_all_from_d, blend_bm25_first, with_doc_name


## Config


In [ ]:
DEV_GOLD_PATH = f"{INPUT_DIR}/dev_300_locked.json"
DEV_CANDIDATES_PATH = f"{INPUT_DIR}/bm25_top100_dev300.json"

# dev150 ĐÃ KHAI TỬ: 256/300 câu của nó giờ nằm trong tập huấn luyện E3.
# dev300 lồng trong dev1000; dev1000 là toà án cuối cho bake-off.

RERANKER_MODEL = "AITeamVN/Vietnamese_Reranker"
DEVICE = "cuda"
K = 5
N_BM25 = 2          # slot giữ cho BM25. Chốt trên 406 câu (dev150+dev300)
USE_DOC_NAME = False # ĐÃ THỬ VÀ THUA: -2.0 điểm (98.8% bootstrap). Slug URL
                     # của D không dấu, câu hỏi có dấu -> nhiễu hơn tín hiệu.
OUTPUT_DIR = "/kaggle/working/outputs"


## Bước 1 -- Đọc dev + candidate đã chunk sẵn (không cần corpus)


In [ ]:
with open(DEV_GOLD_PATH, encoding="utf-8") as f:
    dev_data = json.load(f)
with open(DEV_CANDIDATES_PATH, encoding="utf-8") as f:
    dev_candidates = json.load(f)  # {"qid": [{"doc_id":..,"top_chunks":[...]}]}

dev_questions = {qid: v["question"] for qid, v in dev_data.items()}
dev_gold = {qid: v["answer"] for qid, v in dev_data.items()}
print(f"Dev: {len(dev_data)} câu")

missing = set(dev_questions) - set(dev_candidates)
if missing:
    raise ValueError(f"{len(missing)} câu không có candidate tương ứng")

total_pairs = sum(len(c.get("top_chunks", [])) for qid in dev_candidates for c in dev_candidates[qid])
print(f"Tổng số cặp cần cross-encoder chấm: {total_pairs} (~{total_pairs/len(dev_data):.0f}/câu)")


## Bước 2 -- Load reranker


In [ ]:
score_fn = load_reranker(RERANKER_MODEL, device=DEVICE)


## Bước 3 -- Chấm điểm 1 lượt, dump đủ điểm, so 3 chiến lược

`score_all_from_d` trả về điểm của **cả 100 document**, không chỉ top-5. Cùng một
lượt GPU nhưng mọi thí nghiệm fusion sau đó chạy được trên CPU trong vài giây.


In [ ]:
# 1 lượt GPU cho cấu hình gốc
scores = score_all_from_d(dev_questions, dev_candidates, score_fn)

rank      = lambda S, q: [d for d, _ in sorted(S[q].items(), key=lambda x: -x[1]["ce"])]
bm25_rank = {q: [str(c["doc_id"]) for c in dev_candidates[q]] for q in dev_questions}

def do(S, n):
    return evaluate(dev_gold, {q: blend_bm25_first(rank(S, q), bm25_rank[q], k=K, n_bm25=n)
                               for q in dev_questions}, k=K)["recall"]

predicted_dev = {q: blend_bm25_first(rank(scores, q), bm25_rank[q], k=K, n_bm25=N_BM25)
                 for q in dev_questions}
result = evaluate(dev_gold, predicted_dev, k=K)
print(f"==> N_BM25={N_BM25}  recall@{K}: {result['recall']:.4f}  <== đường chính\n")

print("đối chứng:")
print(f"  BM25 thuần     {evaluate(dev_gold, {q: bm25_rank[q][:K] for q in dev_questions}, k=K)['recall']:.4f}")
for j in (0, 1, 2, 3):
    print(f"  n_bm25={j}       {do(scores, j):.4f}" + ("   (= rerank thuần)" if j == 0 else ""))

# --- Thí nghiệm: ghép tên văn bản vào chunk (task #1) ---
# Lượt GPU THỨ HAI, gấp đôi thời gian. Trường `name` của D là slug KHÔNG DẤU nên
# hiệu quả chưa chắc; đo rồi hãy tin.
if USE_DOC_NAME:
    scores_nm = score_all_from_d(dev_questions, with_doc_name(dev_candidates), score_fn)
    print("\nghép tên văn bản:")
    for j in (0, 1, 2):
        print(f"  n_bm25={j}       {do(scores_nm, j):.4f}")
else:
    scores_nm = None

hc = hard_cases(result)
print(f"\nHard cases: {len(hc)} / {len(dev_data)}")


Xem qua vài ca khó (E4):


In [ ]:
for qid in hc[:5]:
    print(qid, "-", dev_questions[qid])
    print("  gold:", dev_gold[qid], "| predicted:", predicted_dev[qid])


## Bước 4 -- Lưu dump điểm + kết quả

`scores_dev_*.json` là thứ quan trọng nhất: có nó rồi thì tinh chỉnh trọng số fusion
không cần GPU nữa. Nhớ tải về, Kaggle không giữ `/kaggle/working` qua session.


In [ ]:
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
tag = RERANKER_MODEL.replace("/", "_")
n_params = sum(p.numel() for p in score_fn.model.parameters())

with open(f"{OUTPUT_DIR}/scores_dev300_{tag}.json", "w", encoding="utf-8") as f:
    json.dump(scores, f, ensure_ascii=False)
if scores_nm is not None:
    with open(f"{OUTPUT_DIR}/scores_dev300_{tag}_DOCNAME.json", "w", encoding="utf-8") as f:
        json.dump(scores_nm, f, ensure_ascii=False)
with open(f"{OUTPUT_DIR}/dev_eval_{tag}.json", "w", encoding="utf-8") as f:
    json.dump({**result, "model": RERANKER_MODEL, "n_params": n_params,
               "dev": DEV_GOLD_PATH, "n_bm25": N_BM25, "use_doc_name": USE_DOC_NAME},
              f, ensure_ascii=False, indent=2)

print(f"{RERANKER_MODEL}: {n_params:,} tham số ({n_params/1e9:.3f}B)")
for f_ in sorted(os.listdir(OUTPUT_DIR)):
    print(f"  {f_}  {os.path.getsize(os.path.join(OUTPUT_DIR, f_)):,} bytes")
print("\nTẢI TẤT CẢ VỀ MÁY.")
